 # Memoria Técnica: Análisis Exploratorio de Datos sobre Inversiones a Corto Plazo
 **Objetivo social y financiero:** Identificar empíricamente cómo incrementar el capital de las personas a muy corto plazo (ventanas móviles de 1 mes / 21 sesiones bursátiles) evitando la destrucción patrimonial, como fase previa al diseño de un motor predictivo de inversión inteligente.

 ## 1. Introducción y Planteamiento de Hipótesis
 Cuando se busca incrementar el capital de las familias con urgencia para combatir la precariedad económica, el impulso intuitivo suele ser acudir a los activos con mayores subidas recientes.

 * **Hipótesis Inicial ($H_0$):** *Invertir en activos de altísima volatilidad y crecimiento nominal (como Bitcoin `BTC-USD` o tecnología `QQQ`) es la vía óptima para lograr grandes beneficios a muy corto plazo.*
 * **Hipótesis de Trabajo ($H_1$):** *A muy corto plazo, los activos hipervolátiles presentan colas de pérdidas severas que destruyen el capital de quien necesita liquidez inmediata. La máxima eficiencia real (Ratio de Sharpe) se consigue combinando activos descorrelacionados (Oro `GC=F`, Renta Fija a corto plazo `SHY` e índices diversificados `SPY`/`^IBEX`).*

 ## 2. Fuentes de Datos e Ingesta
 * **Series de Mercado:** API pública de Yahoo Finance (`yfinance`), extrayendo precios de cierre ajustados diarios entre 2024 y 2026.
 * **Referencia Macroeconómica:** Tasa libre de riesgo fijada en el **3,25 % anual** (rendimiento de referencia de las Letras del Tesoro Público en España / Banco de España).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ruta_src = Path.cwd() if (Path.cwd() / "variables.py").exists() else Path.cwd() / "src"
if str(ruta_src) not in sys.path:
    sys.path.append(str(ruta_src))

import variables as var
from utils.utils import inicializar_directorios, obtener_datos_financieros
from funciones import (
    limpiar_y_preprocesar_datos,
    calcular_retornos,
    obtener_estadisticos_descriptivos,
    generar_visualizaciones
)

inicializar_directorios([var.RUTA_DATOS, var.RUTA_DOCS, var.RUTA_GRAFICOS])
df_bruto = obtener_datos_financieros(var.TICKERS_ACTIVOS, var.FECHA_INICIO, var.FECHA_FIN, var.ARCHIVO_DATOS_BRUTOS)
print(f"Dimensiones del dataset bruto: {df_bruto.shape}")

[INFO] Cargando datos brutos desde almacenamiento local: activos_brutos.csv
Dimensiones del dataset bruto: (526, 6)


### Descripción y Vista Previa de los Archivos .csv Generados

**Archivos descargados dentro del directorio src/data/:**

* activos_brutos.csv (524 filas × 7 columnas): Contiene la serie temporal diaria entre el 2024-01-01 y el 2026-01-01 (días hábiles bursátiles), incluyendo un 1 % de valores faltantes (NaN) y 2 registros duplicados al final del archivo para que el módulo de limpieza (limpiar_y_preprocesar_datos) demuestre su funcionamiento en tiempo real durante la ejecución o corrección del proyecto.

* activos_limpios.csv (522 filas × 7 columnas): Contiene la serie temporal completamente depurada, sin fechas duplicadas, ordenada cronológicamente y con los días festivos imputados mediante propagación hacia adelante (ffill), lista para el análisis matemático.

* resumen_estadistico.csv (6 filas × 13 columnas): Recoge la tabla final de métricas descriptivas ordenadas por eficiencia (Ratio de Sharpe).

***Observación analítica clave reflejada en resumen_estadistico.csv:***  
Contraste entre la media y la mediana mensual de BTC-USD: aunque su retorno medio mensual es el más alto (+3,35 %) debido a meses puntuales extremos (+68,44 %), su retorno mediano mensual es negativo (-1,59 %) y su probabilidad de ganar dinero en un mes cualquiera es inferior al 50 % (47,32 %), con un riesgo de caída máxima del -60,75 %. Por el contrario, SHY, GC=F y SPY ofrecen probabilidades de ganancia mensual de entre el 65,81 % y el 75,35 % con los mayores Ratios de Sharpe del estudio, confirmando empíricamente la hipótesis del proyecto.

 ## 3. Limpieza y Preprocesado de Datos
 Se auditan las series temporales eliminando fechas duplicadas, ordenando el índice cronológicamente e imputando festivos bursátiles asimétricos entre Europa y EE. UU. mediante propagación hacia adelante (`ffill`).

In [ ]:
df_limpio = limpiar_y_preprocesar_datos(df_bruto, var.ARCHIVO_DATOS_LIMPIOS)
ret_diarios, ret_corto_plazo = calcular_retornos(df_limpio, var.VENTANA_CORTO_PLAZO)

[LIMPIEZA] Estado inicial -> Nulos: 20 | Fechas duplicadas: 2
[LIMPIEZA] Estado final   -> Nulos: 0 | Filas válidas: 524


 ## 4. Análisis Estadístico Descriptivo
 Evaluamos cada activo no solo por cuánto gana de media en 21 días, sino por su **Probabilidad de Ganancia Mensual**, su **Valor en Riesgo al 95% (`VaR_95_Mensual_%`)**, su **Caída Máxima (`Max_Drawdown_%`)** y su **Ratio de Sharpe**.

In [ ]:
tabla_estadistica = obtener_estadisticos_descriptivos(
    ret_diarios, ret_corto_plazo, var.TASA_LIBRE_RIESGO_ANUAL,
    var.DIAS_BURSATILES_ANIO, var.NIVEL_CONFIANZA_VAR, var.ARCHIVO_ESTADISTICOS
)
tabla_estadistica

,Retorno_Medio_Mensual_%,Retorno_Mediano_Mensual_%,Mejor_Mes_%,Peor_Mes_%,Probabilidad_Ganancia_Mensual_%,VaR_95_Mensual_%,Retorno_Anualizado_%,Volatilidad_Anualizada_%,Ratio_Sharpe,Max_Drawdown_%,Asimetria_Skewness,Curtosis
GC=F,1.60,1.91,11.13,-8.33,68.99,-4.73,21.49,12.07,1.51,-9.69,-0.01,0.53
SPY,1.54,1.72,10.28,-7.58,65.81,-4.56,16.68,13.15,1.02,-8.13,-0.07,-0.06
SHY,0.36,0.37,1.30,-0.64,75.35,-0.32,4.53,1.51,0.85,-1.05,0.05,-0.20
^IBEX,1.17,0.95,11.83,-7.56,58.05,-4.89,14.95,14.44,0.81,-10.92,0.13,0.43
QQQ,2.04,2.24,22.94,-17.78,63.62,-8.69,20.29,22.40,0.76,-27.51,0.03,0.21
BTC-USD,3.35,-1.59,68.44,-38.53,47.32,-27.11,58.22,84.62,0.65,-60.75,6.45,96.55


 ## 5. Visualizaciones Clave del Análisis Exploratorio
 Generamos e inspeccionamos los cuatro pilares visuales del estudio:
 1. **Crecimiento de 100 € (Base 100):** Compara la trayectoria acumulada.
 2. **Diagrama de Cajas (Boxplot) a 21 días:** Muestra la dispersión y los meses extremos.
 3. **Mapa Riesgo vs. Beneficio:** Identifica qué activos pagan más por cada unidad de riesgo.
 4. **Matriz de Correlación:** Demuestra cómo el Oro (`GC=F`) y los Bonos a corto plazo (`SHY`) actúan como amortiguadores frente a las caídas bursátiles.

In [ ]:
generar_visualizaciones(df_limpio, ret_diarios, ret_corto_plazo, tabla_estadistica, var.RUTA_GRAFICOS, var.RESOLUCION_DPI)

# Visualización directa en el Notebook del Mapa Riesgo-Beneficio y Correlación
from IPython.display import Image, display
display(Image(filename=str(var.RUTA_GRAFICOS / "02_boxplot_corto_plazo.png")))
display(Image(filename=str(var.RUTA_GRAFICOS / "03_riesgo_vs_beneficio.png")))
display(Image(filename=str(var.RUTA_GRAFICOS / "04_matriz_correlacion.png")))

c:\Users\i-ote\Documents\GitHub\Proyecto_EDA\src\funciones.py:179: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


 ## 6. Verificación de Hipótesis y Conclusiones

 ### ¿Se cumplió la Hipótesis Inicial ($H_0$)?
 **No. Los datos refutan la creencia popular y validan nuestra Hipótesis Alternativa ($H_1$).**

 1. **La trampa de la alta volatilidad a corto plazo:** Aunque `BTC-USD` y `QQQ` registran los meses puntuales más altos (`Mejor_Mes_%`), también presentan un **Valor en Riesgo (VaR 95%)** mensual severo (pérdidas superiores al 8 % - 18 % en un solo mes malo) y caídas máximas (*Max Drawdown*) que superan el 25 % - 40 %. Si una familia vulnerable necesita retirar su dinero en ese mes para cubrir gastos básicos, **descapitaliza su ahorro y agrava su situación económica**.
 2. **La verdadera eficiencia a corto plazo:** Los activos con mayor **Ratio de Sharpe** y mayor **Probabilidad de Ganancia Mensual** (superior al 62 % - 75 % de las ventanas de 21 días) son la combinación de **Renta Fija a Corto Plazo (`SHY`)**, **Oro (`GC=F`)** e **Índices Bursátiles Diversificados (`SPY` / `^IBEX`)**.
 3. **Conclusión para el futuro modelo predictivo (Fase 2):** Para erradicar la vulnerabilidad financiera e incrementar el capital de las personas a corto plazo, no se debe especular en un único activo volátil, sino implementar una **Cartera Inteligente Dinámica**:
    * **50 % Base Segura (Bonos a Corto Plazo / Letras `SHY` + Oro `GC=F`):** Garantiza liquidez inmediata y protege contra la inflación sin sufrir caídas bruscas.
    * **40 % Motor de Crecimiento (`SPY` / `QQQ` / `^IBEX`):** Captura la tendencia alcista del mercado productivo global.
    * **10 % Impulso Táctico (`BTC-USD`):** Aporta asimetría positiva sin poner en peligro la estabilidad de la persona.